
# Entanglement Mobility: Dynamic Slipping vs. Fixed Links

In the first-generation Maximum Entropy Homogenization Procedure (MEHP) Force Balance
(:class:`~pylimer_tools_cpp.MEHPForceBalance`, :cite:t:`bernhard_phantom_2025`), topological
entanglements can be modeled with dynamic mobility along the polymer chain contour.

When an entanglement link couples two strands, it partitions each strand into sub-segments
with fractional contour lengths $\alpha$ and $1 - \alpha$. The simulation method
provides a dedicated parameter to control whether these links are permitted to slide:

- **Dynamic Slip-Links** (``disable_slipping=False``, default in :class:`~pylimer_tools_cpp.MEHPForceBalance`):
  During energy minimization, the link coordinates displace in 3D space *and* the contour
  partitions $\alpha$ are iteratively updated to equalize tensions between adjacent strand segments:

  .. math::

     \kappa_1 \Delta \mathbf{r}_1 = \kappa_2 \Delta \mathbf{r}_2

- **Fixed Entanglement Links** (``disable_slipping=True``):
  The contour partitions $\alpha$ remain frozen at their initially sampled positions.
  The link relaxes in 3D space like a localized 4-functional junction node, but cannot slide
  along either strand contour.

## Why Was Slipping Discarded in Force Balance 2?
In :cite:t:`bernhard_phantom_2025`, Bernhard and Gusev thoroughly evaluated the physical and
computational effects of allowing links to slip along chain contours and found:

1. **Artificial Softening under Affine Deformation**:
   In a real entangled polymer network, entanglements are constrained by surrounding chains
   forming an effective tube. When links are free to slip along the contour without constraint,
   chains dynamically slide through the junctions to relieve tensile stress during macroscopic deformation.
   This leads to an unphysical underestimation of the plateau shear modulus $G$.
2. **Computational Overhead & Slow Convergence**:
   Coupling 3D displacement updates with non-linear contour partition optimizations significantly
   increases the required number of relaxation iterations (often 2-3× more steps) and limits
   numerical stability in large networks.
3. **Unnecessary Complication**:
   Fixing the links at their sampled contour positions ($\alpha = \text{const}$) yielded
   modulus predictions in excellent agreement with experimental plateau moduli and molecular dynamics
   simulations, without the numerical stiffness and computational cost of dynamic slipping.

Consequently, the second-generation :class:`~pylimer_tools_cpp.MEHPForceBalance2` removed contour
slipping entirely, treating entanglements strictly as fixed **entanglement links** (or compliant
Hookean springs).

This example illustrates:

- **Modulus Reduction**: How dynamic contour slipping reduces network stiffness across entanglement densities.
- **Convergence Cost**: The substantial increase in relaxation iterations when slipping is enabled.
- **Statistical Significance**: Multi-seed ensemble sampling with 95% confidence intervals and committed caching.


## Multi-Seed Simulation Procedure
We evaluate :class:`~pylimer_tools_cpp.MEHPForceBalance` with both ``disable_slipping=True``
and ``disable_slipping=False`` across multiple random seeds and entanglement densities.

To keep CI execution and documentation builds fast (< 1 s) while allowing deep statistical
sampling, results are cached into a committed ``.npz`` archive and loaded automatically.
You can rerun the live multi-seed computation at any time by passing ``--recompute`` or
setting ``PYLIMER_RECOMPUTE_CACHE=1``.



In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import sys
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np

# Ensure examples/ is on sys.path when script is executed directly
if "__file__" in globals():
    _examples_dir = str(Path(__file__).resolve().parents[1])
    if _examples_dir not in sys.path:
        sys.path.insert(0, _examples_dir)

from _example_cache import (  # noqa: E402
    compute_ensemble_statistics,
    get_example_cache_path,
    load_or_compute_cache,
    resolve_max_workers,
)
from pylimer_tools.io.bead_spring_parameter_provider import (  # noqa: E402
    ParameterType,
    get_parameters_for_polymer,
)
from pylimer_tools_cpp import MCUniverseGenerator  # noqa: E402


def _simulate_slipping_vs_fixed_single_repeat(
    r: int,
    seed: int,
    box_l: float,
    b: float,
    n_strands: int,
    strand_len: int,
    conversion: float,
    density_factors: Sequence[float],
    nominal_n_ent: int,
    cutoff: float,
    r02: float,
    vol: float,
    gamma_conv: float,
) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Execute a single Monte Carlo realization comparing fixed vs. slipping links."""
    gen = MCUniverseGenerator(box_l, box_l, box_l)
    gen.set_seed(seed)
    gen.set_bead_distance(b)
    gen.set_mean_squared_bead_distance(b**2)
    gen.add_strands(n_strands, [strand_len] * n_strands)
    gen.add_crosslinkers(int(0.5 * n_strands), 4)
    gen.link_strands_to_conversion(conversion)

    n_factors = len(density_factors)
    g_fixed = np.zeros(n_factors)
    g_slip = np.zeros(n_factors)
    iter_fixed = np.zeros(n_factors)
    iter_slip = np.zeros(n_factors)

    for i, factor in enumerate(density_factors):
        n_target = int(factor * nominal_n_ent)

        if n_target == 0:
            # Bare unentangled crosslinker network
            fb = gen.get_force_balance()
            fb.run_force_relaxation(
                max_nr_of_steps=5000,
                disable_slipping=True)
            g_pa = (gamma_conv * np.sum(fb.get_gamma_factors(r02)) / vol) * 1e6
            it = fb.get_nr_of_iterations()
            g_fixed[i] = g_pa
            g_slip[i] = g_pa
            iter_fixed[i] = it
            iter_slip[i] = it
        else:
            # Sample entanglements analytically
            es = gen.sample_entanglements(
                n_target, cutoff, 0.0, 0, 3.0, f"ent_{seed}_{factor}", True
            )

            # 1. Fixed Entanglement Links (disable_slipping=True)
            fb_fixed = gen.get_force_balance(es)
            fb_fixed.run_force_relaxation(
                max_nr_of_steps=5000, disable_slipping=True)
            g_fixed[i] = (
                gamma_conv * np.sum(fb_fixed.get_gamma_factors(r02)) / vol
            ) * 1e6
            iter_fixed[i] = fb_fixed.get_nr_of_iterations()

            # 2. Dynamic Slipping Links (disable_slipping=False)
            fb_slip = gen.get_force_balance(es)
            fb_slip.run_force_relaxation(
                max_nr_of_steps=5000, disable_slipping=False)
            g_slip[i] = (
                gamma_conv * np.sum(fb_slip.get_gamma_factors(r02)) / vol
            ) * 1e6
            iter_slip[i] = fb_slip.get_nr_of_iterations()

    return r, g_fixed, g_slip, iter_fixed, iter_slip


def run_slipping_vs_fixed_simulation(
    density_factors: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    n_strands: int = 10000,
    strand_len: int = 50,
    conversion: float = 0.95,
    n_repeats: int = 10,
    base_seed: int = 42,
    max_workers: Optional[int] = None,
) -> Dict[str, Any]:
    """Evaluate shear modulus and iteration count comparing slipping vs. fixed links."""
    params = get_parameters_for_polymer(
        "PDMS", parameter_type=ParameterType.GAUSSIAN)
    b = params.get("<b>").to(params.get("distance_units")).magnitude
    bead_density = params.get_bead_density()
    cutoff = params.get_sampling_cutoff()
    nominal_ent_density = params.get_entanglement_density()
    r02 = params.get("R02").to(params.get("distance_units") ** 2).magnitude
    kbt = params.get("T") * params.get("kb")
    gamma_conv = (
        (kbt / ((params.get("distance_units")) ** 3)).to("MPa")).magnitude

    n_atoms = n_strands * strand_len + int(0.5 * n_strands)
    vol = n_atoms / bead_density
    box_l = vol ** (1.0 / 3.0)
    nominal_n_ent = int(nominal_ent_density * vol)

    all_g_fixed = np.zeros((len(density_factors), n_repeats))
    all_g_slip = np.zeros((len(density_factors), n_repeats))
    all_iter_fixed = np.zeros((len(density_factors), n_repeats))
    all_iter_slip = np.zeros((len(density_factors), n_repeats))

    tasks = [
        (
            r,
            base_seed + 101 * r,
            box_l,
            b,
            n_strands,
            strand_len,
            conversion,
            density_factors,
            nominal_n_ent,
            cutoff,
            r02,
            vol,
            gamma_conv,
        )
        for r in range(n_repeats)
    ]

    workers = min(resolve_max_workers(max_workers), n_repeats)

    if workers > 1 and n_repeats > 1:
        print(
            f"Executing {n_repeats} repeats in parallel across {workers} worker processes..."
        )
        completed = 0
        with ProcessPoolExecutor(max_workers=workers) as executor:
            futures = [
                executor.submit(
                    _simulate_slipping_vs_fixed_single_repeat, *task)
                for task in tasks
            ]
            for fut in as_completed(futures):
                r, g_fixed, g_slip, iter_fixed, iter_slip = fut.result()
                all_g_fixed[:, r] = g_fixed
                all_g_slip[:, r] = g_slip
                all_iter_fixed[:, r] = iter_fixed
                all_iter_slip[:, r] = iter_slip
                completed += 1
                print(
                    f"  [Slipping vs Fixed] Completed repeat {completed}/{
                        n_repeats
                    } (repeat index {r + 1})..."
                )
    else:
        for task in tasks:
            r, g_fixed, g_slip, iter_fixed, iter_slip = (
                _simulate_slipping_vs_fixed_single_repeat(*task)
            )
            all_g_fixed[:, r] = g_fixed
            all_g_slip[:, r] = g_slip
            all_iter_fixed[:, r] = iter_fixed
            all_iter_slip[:, r] = iter_slip
            print(
                f"  [Slipping vs Fixed] Completed repeat {
                    r + 1}/{n_repeats}...")

    stat_g_fixed = compute_ensemble_statistics(all_g_fixed, axis=1)
    stat_g_slip = compute_ensemble_statistics(all_g_slip, axis=1)
    stat_it_fixed = compute_ensemble_statistics(all_iter_fixed, axis=1)
    stat_it_slip = compute_ensemble_statistics(all_iter_slip, axis=1)

    return {
        "density_factors": np.array(density_factors),
        "n_repeats": np.array(n_repeats),
        "nominal_n_ent": np.array(nominal_n_ent),
        "vol": np.array(vol),
        "g_fixed_mean": stat_g_fixed["mean"],
        "g_fixed_std": stat_g_fixed["std"],
        "g_fixed_sem": stat_g_fixed["sem"],
        "g_fixed_ci": stat_g_fixed["ci"],
        "g_fixed_all": all_g_fixed,
        "g_slip_mean": stat_g_slip["mean"],
        "g_slip_std": stat_g_slip["std"],
        "g_slip_sem": stat_g_slip["sem"],
        "g_slip_ci": stat_g_slip["ci"],
        "g_slip_all": all_g_slip,
        "iter_fixed_mean": stat_it_fixed["mean"],
        "iter_fixed_std": stat_it_fixed["std"],
        "iter_fixed_sem": stat_it_fixed["sem"],
        "iter_fixed_ci": stat_it_fixed["ci"],
        "iter_slip_mean": stat_it_slip["mean"],
        "iter_slip_std": stat_it_slip["std"],
        "iter_slip_sem": stat_it_slip["sem"],
        "iter_slip_ci": stat_it_slip["ci"],
    }

## Loading Cached Data or Recomputing
Resolves the committed cache file or recomputes when requested:



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    cache_path = get_example_cache_path(
        "slipping_vs_fixed_cache.npz", "force_balance")
    data = load_or_compute_cache(cache_path, run_slipping_vs_fixed_simulation)

    density_factors = data["density_factors"]
    n_repeats = int(data.get("n_repeats", 8))

    g_fixed_mean = data["g_fixed_mean"]
    g_fixed_ci = data["g_fixed_ci"]

    g_slip_mean = data["g_slip_mean"]
    g_slip_ci = data["g_slip_ci"]

    iter_fixed_mean = data["iter_fixed_mean"]
    iter_fixed_ci = data["iter_fixed_ci"]

    iter_slip_mean = data["iter_slip_mean"]
    iter_slip_ci = data["iter_slip_ci"]

## Visualizing Modulus Reduction & Computational Cost
The two subplots below highlight the key conclusions of :cite:t:`bernhard_phantom_2025`:

1. **Modulus Underestimation**:
   When links are permitted to slip (``disable_slipping=False``), the predicted equilibrium
   modulus is significantly reduced (by ~30-40% at high entanglement densities) because
   contour sliding allows the chain to relax affine tension. Fixed links (``disable_slipping=True``)
   preserve the effective topological constraint.
2. **Relaxation Iteration Count**:
   Slipping requires solving coupled non-linear partition updates, causing a 2-3× increase in
   iterations required to reach the force tolerance threshold.



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    fig, (ax_mod, ax_iter) = plt.subplots(1, 2, figsize=(11, 4.5))

    # Modulus comparison
    ax_mod.plot(
        density_factors,
        g_fixed_mean,
        "s-",
        color="#2b83ba",
        linewidth=2,
        markersize=6,
        label=f"Fixed Links (disable_slipping=True, N={n_repeats})",
    )
    ax_mod.fill_between(
        density_factors,
        g_fixed_mean - g_fixed_ci,
        g_fixed_mean + g_fixed_ci,
        color="#2b83ba",
        alpha=0.2,
        label="95% Confidence Interval",
    )
    ax_mod.errorbar(
        density_factors,
        g_fixed_mean,
        yerr=g_fixed_ci,
        fmt="none",
        ecolor="#2b83ba",
        capsize=4,
        elinewidth=1.2,
    )

    ax_mod.plot(
        density_factors,
        g_slip_mean,
        "o--",
        color="#d7191c",
        linewidth=2,
        markersize=6,
        label=f"Slipping Links (disable_slipping=False, N={n_repeats})",
    )
    ax_mod.fill_between(
        density_factors,
        g_slip_mean - g_slip_ci,
        g_slip_mean + g_slip_ci,
        color="#d7191c",
        alpha=0.2,
    )
    ax_mod.errorbar(
        density_factors,
        g_slip_mean,
        yerr=g_slip_ci,
        fmt="none",
        ecolor="#d7191c",
        capsize=4,
        elinewidth=1.2,
    )

    ax_mod.set_xlabel(
        r"Entanglement Density Factor ($N_{\mathrm{ent}} / N_{\mathrm{nominal}}$)"
    )
    ax_mod.set_ylabel(r"Shear Modulus $G$ [Pa]")
    ax_mod.set_title("Equilibrium Shear Modulus vs. Entanglement Density")
    ax_mod.grid(True, linestyle="--", alpha=0.5)
    ax_mod.legend(frameon=True)

    # Iteration count comparison
    ax_iter.plot(
        density_factors,
        iter_fixed_mean,
        "s-",
        color="#2b83ba",
        linewidth=2,
        markersize=6,
        label=r"Fixed Links (disable_slipping=True)",
    )
    ax_iter.fill_between(
        density_factors,
        iter_fixed_mean - iter_fixed_ci,
        iter_fixed_mean + iter_fixed_ci,
        color="#2b83ba",
        alpha=0.2,
    )

    ax_iter.plot(
        density_factors,
        iter_slip_mean,
        "o--",
        color="#d7191c",
        linewidth=2,
        markersize=6,
        label=r"Slipping Links (disable_slipping=False)",
    )
    ax_iter.fill_between(
        density_factors,
        iter_slip_mean - iter_slip_ci,
        iter_slip_mean + iter_slip_ci,
        color="#d7191c",
        alpha=0.2,
    )

    ax_iter.set_xlabel(
        r"Entanglement Density Factor ($N_{\mathrm{ent}} / N_{\mathrm{nominal}}$)"
    )
    ax_iter.set_ylabel("Iterations to Convergence")
    ax_iter.set_title("Convergence Iterations vs. Entanglement Density")
    ax_iter.grid(True, linestyle="--", alpha=0.5)
    ax_iter.legend(frameon=True)

    plt.tight_layout()
    plt.show()

## Summary & Method Selection

.. list-table::
   :widths: 25 35 40
   :header-rows: 1

   * - Feature
     - Fixed Links (``disable_slipping=True``)
     - Dynamic Slipping (``disable_slipping=False``)
   * - **Contour Partition**
     - Frozen at initial sampled position ($\alpha = \text{const}$).
     - Dynamically updated to balance tension along strands.
   * - **Modulus Prediction**
     - Accurately captures experimental plateau modulus (:cite:t:`bernhard_phantom_2025`).
     - Artificially softens modulus due to affine tension release.
   * - **Convergence Speed**
     - Fast; 2-3× fewer iterations.
     - Slower; non-linear coupled partition optimization.
   * - **Recommended Implementation**
     - Use :class:`~pylimer_tools_cpp.MEHPForceBalance2` (links are fixed by design).
     - Historical curiosity in :class:`~pylimer_tools_cpp.MEHPForceBalance`.

